# Embeddings & Transfer Learning — a worked example

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/SCAI-4EUWorkshopAIinMedicineWorkshop/blob/main/Hands-On-Session-3/text_embeddings_example.ipynb)

## Objective

Models only understand **numbers** — yet so much scientific signal lives in **text**:
clinical notes, paper abstracts, even molecules and DNA written as strings. Turning that
text into something a model can learn from means walking one pipeline:

> **Tokenize → Embed → Explore → Classify → Evaluate**

```
  Raw text  →  Tokens  →  Token IDs  →  Embeddings  →  Classifier  →  Prediction
 (a string)  (subwords)  (integers)   (dense vectors)   (trained)
```

We will:

1. **Tokenize** text and see how it becomes integer IDs.
2. Turn those IDs into **embeddings** — dense vectors where *meaning becomes geometry*.
3. Use a **pre-trained** model to embed real documents, then **explore** the space.
4. **Transfer-learn**: embed data with a pre-trained model, then train a small classifier on
   top — the recipe behind the competition. To show it generalizes, we run it on a *different*
   domain: predicting whether a drug **molecule** can cross the blood-brain barrier.

**Key idea:** a model pre-trained on huge corpora already encodes meaning. You rarely train
from scratch — you embed your data with it and put a light classifier on top.

## 1. Setup

> 💡 **Enable the GPU first:** this notebook embeds documents with a pre-trained model, which is much faster on a GPU. In Colab go to **Runtime → Change runtime type → Hardware accelerator → GPU**, then run the cells top to bottom. (It still runs on CPU — just slower.)

In [ ]:
!pip install -q transformers sentence-transformers scikit-learn matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

rng = np.random.default_rng(0)
print("Setup complete!")

## 2. From Text to Tokens

Before anything else, text is **tokenized** — split into units. There are three strategies:

| Strategy | "unhappiness" becomes | Trade-off |
|---|---|---|
| **Character** | `['u','n','h','a','p','p','i','n','e','s','s']` | Tiny vocab, but very long sequences |
| **Word** | `['unhappiness']` | Meaningful, but huge vocab — can't handle new words |
| **Subword** | `['un', 'happiness']` | **Balanced** — frequent words stay whole, rare words split |

**Modern models use subword tokenization.** Frequent words ("the", "cell") stay whole; rare
words ("electroencephalography") are split into known pieces — so *any* word can be encoded.
Different tokenizers make different choices (e.g. case): let's compare two.

In [ ]:
from transformers import AutoTokenizer

tok_gpt2 = AutoTokenizer.from_pretrained("gpt2")              # BPE, case-sensitive
tok_bert = AutoTokenizer.from_pretrained("bert-base-uncased") # WordPiece, lowercases

print(f"{'Tokenizer':<16}{'Vocab size':>12}{'  Case':<12}")
print("-" * 42)
print(f"{'GPT-2':<16}{len(tok_gpt2):>12,}  Preserved")
print(f"{'BERT uncased':<16}{len(tok_bert):>12,}  Lowercased")

text = "The CRISPR-Cas9 system edits DNA."
print(f"\nText: {text!r}\n")
print(f"  GPT-2 ({len(tok_gpt2.tokenize(text)):2}): {tok_gpt2.tokenize(text)}")
print(f"  BERT  ({len(tok_bert.tokenize(text)):2}): {tok_bert.tokenize(text)}")

Notice the markers tokenizers add: GPT-2's `Ġ` means "space before this token", BERT's
`##` means "continues the previous word". BERT also **lowercases** ("DNA" → "dna"), losing
case that can matter for scientific text. Each token then maps to an integer **token ID** from
the model's fixed vocabulary:

```
"Machine" → 33423      "learning" → 4673
```

**But token IDs are just arbitrary integers** — ID 33423 is no more "similar" to 33424 than
to 5. They carry no meaning on their own. That is exactly what embeddings fix.

## 3. Embeddings: Meaning Becomes Geometry

The key idea of modern NLP: **convert tokens into vectors where similar things are close
together.** The naive way — *one-hot* encoding — fails immediately:

In [ ]:
# One-hot: each word is a vector with a single 1. Every word is equally far from every other.
one_hot = {"cat": [1,0,0,0,0], "dog": [0,1,0,0,0], "tree": [0,0,0,0,1]}
one_hot = {k: np.array(v) for k, v in one_hot.items()}

print("One-hot distances:")
print(f"  cat <-> dog : {np.linalg.norm(one_hot['cat'] - one_hot['dog']):.2f}")
print(f"  cat <-> tree: {np.linalg.norm(one_hot['cat'] - one_hot['tree']):.2f}")
print("  -> identical! one-hot loses the fact that cat and dog are both animals.\n")

# Dense embeddings: each dimension captures some aspect of meaning (here, hand-picked).
#                    [is_animal, has_fur, can_move, is_pet]
dense = {"cat":  np.array([0.9, 0.9, 0.9, 0.9]),
         "dog":  np.array([0.9, 0.9, 0.9, 0.9]),
         "tree": np.array([0.9, 0.0, 0.0, 0.0])}
print("Dense-embedding distances:")
print(f"  cat <-> dog : {np.linalg.norm(dense['cat'] - dense['dog']):.2f}  (close - both pets)")
print(f"  cat <-> tree: {np.linalg.norm(dense['cat'] - dense['tree']):.2f}  (far - different things)")

To **measure** closeness between embeddings we use **cosine similarity** — the angle
between two vectors, ranging from -1 (opposite) to +1 (same direction):

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\lVert A \rVert\, \lVert B \rVert}$$

It cares about *direction*, not magnitude, so `[1,2,3]` and `[2,4,6]` score 1.0.

In [ ]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"cosine(cat, dog)  = {cosine_sim(dense['cat'],  dense['dog']):+.2f}")
print(f"cosine(cat, tree) = {cosine_sim(dense['cat'],  dense['tree']):+.2f}")

Real embeddings aren't hand-designed — a model **learns** them. In PyTorch an embedding
layer (`nn.Embedding`) is just a learnable lookup table: token ID → dense vector, random at
first, nudged during training so tokens used in similar contexts end up close. Training those
representations well takes enormous data and compute, so in practice we **reuse a pre-trained
model**.

## 4. Pre-trained Sentence Embeddings

A pre-trained model gives an embedding per **token**; to embed a whole sentence we **pool**
them (e.g. average). [Sentence Transformers](https://www.sbert.net/) package this up:
`model.encode(text)` returns one vector per text, trained so that similar sentences land close.

In [ ]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2: small, fast, trained on 1B+ sentence pairs -> 384-dim vectors.
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "The cat is sleeping on the couch.",
    "A dog is resting on the sofa.",
    "Machine learning turns data into insights.",
    "Deep learning models can recognize images.",
    "The weather is sunny today.",
]
emb = embedder.encode(sentences)
print(f"Each sentence -> {emb.shape[1]}-dim vector   (encoded {emb.shape[0]} sentences)")

In [ ]:
def plot_similarity(matrix, labels, title, cmap="Blues"):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{matrix[i,j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(title); fig.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

labels = [s[:22] + "..." for s in sentences]
plot_similarity(cosine_similarity(emb), labels, "Sentence similarity (all-MiniLM-L6-v2)")

The two pet sentences score high together, as do the two ML sentences — even though they
share few exact words ("couch" ≈ "sofa", "sleeping" ≈ "resting"). The model captures *meaning*,
not keywords.

### Beyond natural language

The very same pipeline works on any data you can write as a string. Specialized models are
pre-trained on each domain's "language":

| Domain | Data written as text | Pre-trained model |
|---|---|---|
| **Text** | abstracts, clinical notes | SciBERT, MiniLM |
| **Molecules** | SMILES (`CC(=O)Oc1ccccc1C(=O)O`) | ChemBERTa |
| **Proteins** | amino-acid sequences (`MSKIIE…`) | ESM-2 |
| **DNA** | nucleotides (`ATCGATCG…`) | Nucleotide Transformer |

For example, a chemistry tokenizer splits a molecule's SMILES string into atoms and bonds —
exactly like subword tokenization, but for chemistry:

In [ ]:
# A molecule is just text too: ChemBERTa tokenizes SMILES into chemical sub-units.
chem_tok = AutoTokenizer.from_pretrained("seyonec/ChemBERTa-zinc-base-v1")
for name, smiles in [("Ethanol", "CCO"), ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O")]:
    print(f"{name:8} {smiles:28} -> {chem_tok.tokenize(smiles)}")

## 5. A Real Dataset — Explore the Embedding Space

Let's embed actual documents. We use **20 Newsgroups** (built into scikit-learn): forum posts
labelled by topic. We keep four distinct topics so the structure is easy to see, and strip
headers/signatures so the model must rely on the *content*.

In [ ]:
categories = ["sci.med", "sci.space", "rec.autos", "comp.graphics"]
raw = fetch_20newsgroups(subset="train", categories=categories,
                         remove=("headers", "footers", "quotes"))

# Subsample to keep embedding fast on CPU.
idx = rng.choice(len(raw.data), size=800, replace=False)
docs   = [raw.data[i] for i in idx]
labels_y = raw.target[idx]
class_names = raw.target_names

print(f"{len(docs)} documents across {len(class_names)} topics: {class_names}")
print("counts:", {class_names[c]: int((labels_y == c).sum()) for c in range(len(class_names))})
print(f"\nExample ({class_names[labels_y[0]]}):\n{docs[0][:250].strip()}...")

In [ ]:
# Embed every document once with the pre-trained model -> reuse for viz, search, and classification.
X = embedder.encode(docs, batch_size=64, show_progress_bar=True)
print(f"\nEmbedding matrix: {X.shape[0]} documents x {X.shape[1]} features")

### Visualize the space (PCA)

The embeddings live in 384 dimensions. Project them to 2D with **PCA** and colour by topic —
documents about the same subject should cluster, with no labels given to the embedder.

In [ ]:
coords = PCA(n_components=2).fit_transform(X)

plt.figure(figsize=(8, 6))
for c, name in enumerate(class_names):
    m = labels_y == c
    plt.scatter(coords[m, 0], coords[m, 1], s=14, alpha=0.6, label=name)
plt.legend(); plt.title("Document embeddings (PCA to 2D)")
plt.xlabel("PC 1"); plt.ylabel("PC 2")
plt.tight_layout(); plt.show()

### Application: semantic search

Because meaning is geometry, **search** becomes "find the nearest vectors" — no keyword
matching needed. Embed a query, rank documents by cosine similarity:

In [ ]:
query = "how to treat high blood pressure"
q = embedder.encode([query])
scores = cosine_similarity(q, X)[0]

print(f"Query: {query!r}\n")
for rank, i in enumerate(scores.argsort()[::-1][:3], 1):
    print(f"{rank}. [{class_names[labels_y[i]]}, sim={scores[i]:.2f}] {docs[i][:90].strip()}...")

## 6. Transfer Learning: the Same Recipe on Another Domain

Everything so far was *text*. The payoff is that the **exact same recipe** — embed with a
pre-trained model, train a light classifier on top — works on any data you can write as a
string. This is the idea behind the competition, so let's make it concrete on a **different
domain** to prove the point: **molecules**.

We predict whether a drug can cross the **blood-brain barrier (BBB)** — essential for any drug
that must act in the brain, yet blocked for ~98% of small molecules. Each molecule is a SMILES
string; we embed it with **ChemBERTa** (pre-trained on 77M molecules) and classify:

```
SMILES  →  ChemBERTa (FROZEN)  →  768-dim embedding  →  classifier (TRAINED)  →  crosses BBB?
```

In [ ]:
import pandas as pd

bbbp = pd.read_csv("https://raw.githubusercontent.com/racousin/ai_for_sciences/main/day2/data/molecules_bbbp.csv")
baseline = bbbp["label"].mean()   # label: 1 = permeable, 0 = impermeable

print(f"{len(bbbp)} molecules with measured BBB permeability")
print(bbbp["label_name"].value_counts().to_string())
print(f"\nBaseline (always predict 'permeable'): {baseline:.1%} accuracy")
bbbp.head(3)

In [ ]:
from transformers import AutoModel
import torch

# Reuse the ChemBERTa tokenizer from section 4; load the matching model (frozen).
chem_model = AutoModel.from_pretrained("seyonec/ChemBERTa-zinc-base-v1").eval()

def embed_smiles(smiles, batch_size=32):
    # SMILES strings -> ChemBERTa [CLS] embeddings (no training)
    out = []
    for i in range(0, len(smiles), batch_size):
        enc = chem_tok(smiles[i:i+batch_size], return_tensors="pt",
                       padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            out.append(chem_model(**enc).last_hidden_state[:, 0, :].numpy())  # [CLS] token
    return np.vstack(out)

Xm = embed_smiles(bbbp["SMILES"].tolist())
ym = bbbp["label"].values
print(f"Embedding matrix: {Xm.shape[0]} molecules x {Xm.shape[1]} features")

Same two models as before — start with **logistic regression** as the bar, then an
**MLP** (a hidden layer + non-linear activations) for non-linear structure:

In [ ]:
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    Xm, ym, test_size=0.2, random_state=42, stratify=ym)

bbb_lr  = LogisticRegression(max_iter=2000).fit(Xm_tr, ym_tr)
bbb_mlp = MLPClassifier(hidden_layer_sizes=(256,), max_iter=300, random_state=42).fit(Xm_tr, ym_tr)

for name, clf in [("Logistic Regression", bbb_lr), ("MLP (1 hidden layer)", bbb_mlp)]:
    acc = clf.score(Xm_te, ym_te)
    f1  = f1_score(ym_te, clf.predict(Xm_te), average="macro")
    print(f"{name:<22} accuracy={acc:.1%}  F1-macro={f1:.3f}")
print(f"{'Baseline':<22} accuracy={baseline:.1%}")

## 7. Evaluate

In drug discovery, **not all errors cost the same**: a *false negative* (we predict "blocked"
but the drug actually crosses) could throw away a good candidate. The **classification report**
and **confusion matrix** expose this per class — and you can shift the 0.5 decision threshold to
trade recall for precision when one error is costlier than the other.

In [ ]:
labels_bbb = ["impermeable", "permeable"]
y_pred_m = bbb_mlp.predict(Xm_te)
print(classification_report(ym_te, y_pred_m, target_names=labels_bbb))

fig, ax = plt.subplots(figsize=(5.5, 4.8))
ConfusionMatrixDisplay.from_predictions(
    ym_te, y_pred_m, display_labels=labels_bbb, cmap="Blues", colorbar=False, ax=ax)
ax.set_title("BBB classifier — confusion matrix")
plt.tight_layout(); plt.show()

In [ ]:
# Apply the trained classifier to brand-new molecules.
test_drugs = [
    ("Caffeine",     "Cn1cnc2c1c(=O)n(c(=O)n2C)C",                  "should cross (CNS stimulant)"),
    ("Diazepam",     "CN1C(=O)CN=C(c2ccccc2)c3cc(Cl)ccc13",         "should cross (sedative)"),
    ("Ibuprofen",    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",                  "should NOT cross (peripheral)"),
    ("Penicillin G", "CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",  "should NOT cross (antibiotic)"),
]
for name, smiles, expected in test_drugs:
    prob = bbb_mlp.predict_proba(embed_smiles([smiles]))[0, 1]
    print(f"{name:<13} P(permeable)={prob:5.1%}  ->  {'crosses' if prob > 0.5 else 'blocked':<8} ({expected})")

## 8. Summary

| Step | Technique | Why |
|---|---|---|
| Tokenize | subword tokenizer | Turn data (text or SMILES) into integer IDs |
| Embed | pre-trained model (frozen) | IDs → dense vectors where meaning = geometry |
| Measure | cosine similarity | Compare meanings by angle, not keywords |
| Explore | PCA, similarity matrix, search | See structure; retrieve by meaning |
| Classify | LogReg / MLP on embeddings | Light model on frozen features — transfer learning |
| Evaluate | F1-macro, confusion matrix | Fair across classes; shows *where* it fails |

```
data (text · SMILES · sequence) → tokenize → embed (pre-trained, frozen) → small classifier → evaluate
```

**Key insight:** with embeddings, the heavy lifting is already done by the pre-trained model.
A simple classifier on top of good embeddings beats a complex model trained from scratch on
little data — and the *same two steps* work across domains: here we explored **text** and
classified **molecules** (BBB permeability) with one recipe. Swap the pre-trained model
(SciBERT, ESM-2, Nucleotide Transformer…) and it carries over to proteins, DNA, and beyond.

**Next:** apply this exact recipe to the **clinical-note-triage competition** — embed each
note with a sentence model (as in sections 1–5), train a classifier on top (as in section 6)
to predict the specialty, write `submission.csv`, and climb the F1-macro leaderboard.